# Ablation Study Results Visualization

This notebook visualizes and analyzes the results from the ablation study.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

%matplotlib inline

## 1. Load Ablation Study Results

In [ ]:
# Load results (assuming you've run the ablation study)
results_path = '../ablation_results/ablation_results.csv'

# Check if results exist
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    print("Ablation Study Results:")
    print(df.to_string(index=False))
else:
    print("Results not found. Run: python scripts/ablation_study.py --config configs/baseline.yaml")
    # Create sample data for demonstration
    df = pd.DataFrame({
        'Model': ['Baseline', 'V1-DataAug', 'V2-LossFn', 'V3-Architecture', 'V4-Training', 'V5-Lightweight'],
        'mAP@0.5': [0.753, 0.785, 0.792, 0.810, 0.798, 0.765],
        'mAP@0.75': [0.652, 0.681, 0.693, 0.715, 0.698, 0.660],
        'F1': [0.78, 0.81, 0.82, 0.84, 0.83, 0.79],
        'Precision': [0.80, 0.83, 0.84, 0.86, 0.85, 0.81],
        'Recall': [0.76, 0.79, 0.80, 0.82, 0.81, 0.77],
        'Inference_Time_ms': [15.2, 15.5, 15.8, 18.3, 15.3, 10.5],
        'Model_Size_MB': [6.2, 6.2, 6.2, 12.4, 6.2, 3.8]
    })
    print("\nUsing sample data for demonstration.")
    print(df)

## 2. Performance Comparison

In [ ]:
# Plot mAP comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df))
width = 0.35

ax.bar(x - width/2, df['mAP@0.5'], width, label='mAP@0.5', alpha=0.8)
ax.bar(x + width/2, df['mAP@0.75'], width, label='mAP@0.75', alpha=0.8)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('mAP', fontsize=12)
ax.set_title('mAP Comparison Across All Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Accuracy vs Efficiency Trade-off

In [ ]:
# Scatter plot: Inference Time vs mAP
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Time vs mAP
ax1.scatter(df['Inference_Time_ms'], df['mAP@0.5'], s=200, alpha=0.6, c=range(len(df)), cmap='viridis')
for i, model in enumerate(df['Model']):
    ax1.annotate(model, (df['Inference_Time_ms'].iloc[i], df['mAP@0.5'].iloc[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)
ax1.set_xlabel('Inference Time (ms)', fontsize=12)
ax1.set_ylabel('mAP@0.5', fontsize=12)
ax1.set_title('Inference Time vs mAP Trade-off', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Size vs mAP
ax2.scatter(df['Model_Size_MB'], df['mAP@0.5'], s=200, alpha=0.6, c=range(len(df)), cmap='plasma')
for i, model in enumerate(df['Model']):
    ax2.annotate(model, (df['Model_Size_MB'].iloc[i], df['mAP@0.5'].iloc[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)
ax2.set_xlabel('Model Size (MB)', fontsize=12)
ax2.set_ylabel('mAP@0.5', fontsize=12)
ax2.set_title('Model Size vs mAP Trade-off', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Relative Improvement Analysis

In [ ]:
# Calculate improvements relative to baseline
baseline_map = df[df['Model'] == 'Baseline']['mAP@0.5'].values[0]
baseline_time = df[df['Model'] == 'Baseline']['Inference_Time_ms'].values[0]

df['mAP_Improvement_%'] = ((df['mAP@0.5'] - baseline_map) / baseline_map * 100).round(2)
df['Time_Change_%'] = ((df['Inference_Time_ms'] - baseline_time) / baseline_time * 100).round(2)

# Plot improvements
fig, ax = plt.subplots(figsize=(12, 6))
improved_models = df[df['Model'] != 'Baseline']

x = np.arange(len(improved_models))
width = 0.35

bars1 = ax.bar(x - width/2, improved_models['mAP_Improvement_%'], width, 
               label='mAP Improvement', color='green', alpha=0.7)
bars2 = ax.bar(x + width/2, improved_models['Time_Change_%'], width, 
               label='Time Overhead', color='red', alpha=0.7)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Change (%)', fontsize=12)
ax.set_title('Relative Improvement vs Baseline', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(improved_models['Model'], rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRelative Improvements:")
print(improved_models[['Model', 'mAP_Improvement_%', 'Time_Change_%']].to_string(index=False))

## 5. Radar Chart for Comprehensive Comparison

In [ ]:
from math import pi

# Normalize metrics to 0-1 range for radar chart
metrics = ['mAP@0.5', 'mAP@0.75', 'F1', 'Precision', 'Recall']
df_norm = df[metrics].copy()
for col in metrics:
    df_norm[col] = (df_norm[col] - df_norm[col].min()) / (df_norm[col].max() - df_norm[col].min())

# Create radar chart
angles = [n / float(len(metrics)) * 2 * pi for n in range(len(metrics))]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

for idx, model in enumerate(df['Model']):
    values = df_norm.iloc[idx].values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.set_title('Comprehensive Performance Comparison', size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. Summary and Recommendations

In [ ]:
# Find best models for different criteria
best_accuracy = df.loc[df['mAP@0.5'].idxmax()]
best_speed = df.loc[df['Inference_Time_ms'].idxmin()]
smallest_size = df.loc[df['Model_Size_MB'].idxmin()]

# Calculate efficiency score (accuracy / time)
df['Efficiency_Score'] = df['mAP@0.5'] / (df['Inference_Time_ms'] / 10)
best_efficiency = df.loc[df['Efficiency_Score'].idxmax()]

print("="*60)
print("SUMMARY AND RECOMMENDATIONS")
print("="*60)
print(f"\nBest Accuracy: {best_accuracy['Model']}")
print(f"  mAP@0.5: {best_accuracy['mAP@0.5']:.3f}")
print(f"  Inference Time: {best_accuracy['Inference_Time_ms']:.1f} ms")

print(f"\nFastest Model: {best_speed['Model']}")
print(f"  Inference Time: {best_speed['Inference_Time_ms']:.1f} ms")
print(f"  mAP@0.5: {best_speed['mAP@0.5']:.3f}")

print(f"\nSmallest Model: {smallest_size['Model']}")
print(f"  Size: {smallest_size['Model_Size_MB']:.1f} MB")
print(f"  mAP@0.5: {smallest_size['mAP@0.5']:.3f}")

print(f"\nBest Efficiency: {best_efficiency['Model']}")
print(f"  Efficiency Score: {best_efficiency['Efficiency_Score']:.3f}")
print(f"  mAP@0.5: {best_efficiency['mAP@0.5']:.3f}")
print(f"  Inference Time: {best_efficiency['Inference_Time_ms']:.1f} ms")

print("\n" + "="*60)
print("DEPLOYMENT RECOMMENDATIONS")
print("="*60)
print("\n• For maximum accuracy (research/server): Use", best_accuracy['Model'])
print("• For edge devices (embedded): Use", best_speed['Model'])
print("• For balanced deployment (production): Use", best_efficiency['Model'])
print("="*60)